In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Qwen2.5-14B-Instruct project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


## Qwen-14B-Instruct on Qwen 72B

In [3]:
pip install torch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-14B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


/home/yuexing/miniconda/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█| 579/579 [00:03<00:00, 146.49it/s, Materializing param=model.norm.we
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [2]:
import pandas as pd 
import re
import torch

# Load data
df = pd.read_csv(paths.DATA / "Qwen14B_annotated_MedPAIR_relevancy.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None

# Create results dataframe
results = []

# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["Qwen14B_High_Relevance"]
        question = row["question_options_x"]
        
        # Improved prompt with clearer instructions
        query_full = (
            "You are a clinical reasoning assistant. You will receive a patient case summary "
            "and a multiple-choice question.\n\n"
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Please select the single most appropriate answer. Respond only in the following format:\n\n"
            "Answer: <LETTER>"
    )
    
        # Generate prediction using the model
        inputs = tokenizer(query_full, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
        )

        # Decode the generated response
        raw_response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # Extract the answer letter using improved function
        extracted_answer = extract_answer_letter(raw_response)
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
        qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(paths.PREDICTIONS / "[SR]_Qwen_14B_predictions_14Bprogress.csv", index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "[SR]_Qwen_14B_predictions_Qwen14B.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")


Columns in dataset:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options_x', 'answer_corr', 'ID', 'centaur_question', 'sentence_number', 'answer', 'data_source', 'step1_excerpts', 'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2', 'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7', 'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12', 'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16', 'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20', 'sentence_21', 'Qwen14B_answer', 'Qwen14B_raw_response', 'q1', 'q2', 'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13', 'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'label_21', 'Qwen14B_High_Relevance', 'Match?', 'data_source_corr_trainee']
Processing 1303 rows...
Processing row 1/1303...
✅ Processed Merge Q1: Answer = D
Processing row 2/1303...
✅ Processed Merge Q2: Answer = D
Processing row 3/1303...
✅ Processed Merge Q3: Answ

✅ Processed Merge Q117: Answer = C
Processing row 118/1303...
✅ Processed Merge Q118: Answer = D
Processing row 119/1303...
✅ Processed Merge Q119: Answer = B
Processing row 120/1303...
✅ Processed Merge Q120: Answer = E
Saved progress to CSV after 120 items
Processing row 121/1303...
✅ Processed Merge Q121: Answer = B
Processing row 122/1303...
✅ Processed Merge Q122: Answer = A
Processing row 123/1303...
✅ Processed Merge Q123: Answer = B
Processing row 124/1303...
✅ Processed Merge Q124: Answer = D
Processing row 125/1303...
✅ Processed Merge Q125: Answer = D
Processing row 126/1303...
✅ Processed Merge Q126: Answer = D
Processing row 127/1303...
✅ Processed Merge Q127: Answer = G
Processing row 128/1303...
✅ Processed Merge Q128: Answer = D
Processing row 129/1303...
✅ Processed Merge Q129: Answer = A
Processing row 130/1303...
✅ Processed Merge Q130: Answer = C
Saved progress to CSV after 130 items
Processing row 131/1303...
✅ Processed Merge Q131: Answer = D
Processing row 132/13

✅ Processed Merge Q242: Answer = B
Processing row 243/1303...
✅ Processed Merge Q243: Answer = B
Processing row 244/1303...
✅ Processed Merge Q244: Answer = B
Processing row 245/1303...
✅ Processed Merge Q245: Answer = B
Processing row 246/1303...
✅ Processed Merge Q246: Answer = C
Processing row 247/1303...
✅ Processed Merge Q247: Answer = B
Processing row 248/1303...
✅ Processed Merge Q248: Answer = D
Processing row 249/1303...
✅ Processed Merge Q249: Answer = A
Processing row 250/1303...
✅ Processed Merge Q250: Answer = D
Saved progress to CSV after 250 items
Processing row 251/1303...
✅ Processed Merge Q251: Answer = B
Processing row 252/1303...
✅ Processed Merge Q252: Answer = B
Processing row 253/1303...
✅ Processed Merge Q253: Answer = F
Processing row 254/1303...
✅ Processed Merge Q254: Answer = D
Processing row 255/1303...
✅ Processed Merge Q255: Answer = A
Processing row 256/1303...
✅ Processed Merge Q256: Answer = D
Processing row 257/1303...
✅ Processed Merge Q257: Answer =

✅ Processed Merge Q365: Answer = D
Processing row 366/1303...
✅ Processed Merge Q366: Answer = H
Processing row 367/1303...
✅ Processed Merge Q367: Answer = B
Processing row 368/1303...
✅ Processed Merge Q368: Answer = D
Processing row 369/1303...
✅ Processed Merge Q369: Answer = C
Processing row 370/1303...
✅ Processed Merge Q370: Answer = D
Saved progress to CSV after 370 items
Processing row 371/1303...
✅ Processed Merge Q371: Answer = D
Processing row 372/1303...
✅ Processed Merge Q372: Answer = D
Processing row 373/1303...
✅ Processed Merge Q373: Answer = D
Processing row 374/1303...
✅ Processed Merge Q374: Answer = C
Processing row 375/1303...
✅ Processed Merge Q375: Answer = E
Processing row 376/1303...
✅ Processed Merge Q376: Answer = D
Processing row 377/1303...
✅ Processed Merge Q377: Answer = C
Processing row 378/1303...
✅ Processed Merge Q378: Answer = C
Processing row 379/1303...
✅ Processed Merge Q379: Answer = D
Processing row 380/1303...
✅ Processed Merge Q380: Answer =

✅ Processed Merge Q488: Answer = C
Processing row 489/1303...
✅ Processed Merge Q489: Answer = C
Processing row 490/1303...
✅ Processed Merge Q490: Answer = C
Saved progress to CSV after 490 items
Processing row 491/1303...
✅ Processed Merge Q491: Answer = C
Processing row 492/1303...
✅ Processed Merge Q492: Answer = C
Processing row 493/1303...
✅ Processed Merge Q493: Answer = C
Processing row 494/1303...
✅ Processed Merge Q494: Answer = C
Processing row 495/1303...
✅ Processed Merge Q495: Answer = A
Processing row 496/1303...
✅ Processed Merge Q496: Answer = D
Processing row 497/1303...
✅ Processed Merge Q497: Answer = A
Processing row 498/1303...
✅ Processed Merge Q498: Answer = C
Processing row 499/1303...
✅ Processed Merge Q499: Answer = J
Processing row 500/1303...
✅ Processed Merge Q500: Answer = D
Saved progress to CSV after 500 items
Processing row 501/1303...
✅ Processed Merge Q501: Answer = C
Processing row 502/1303...
✅ Processed Merge Q502: Answer = C
Processing row 503/13

...

✅ Processed Merge Q723: Answer = C
Processing row 724/1303...
✅ Processed Merge Q724: Answer = B
Processing row 725/1303...
✅ Processed Merge Q725: Answer = C
Processing row 726/1303...
✅ Processed Merge Q726: Answer = C
Processing row 727/1303...
✅ Processed Merge Q727: Answer = D
Processing row 728/1303...
✅ Processed Merge Q728: Answer = F
Processing row 729/1303...
✅ Processed Merge Q729: Answer = D
Processing row 730/1303...
✅ Processed Merge Q730: Answer = C
Saved progress to CSV after 730 items
Processing row 731/1303...
✅ Processed Merge Q731: Answer = C
Processing row 732/1303...
✅ Processed Merge Q732: Answer = C
Processing row 733/1303...
✅ Processed Merge Q733: Answer = G
Processing row 734/1303...
✅ Processed Merge Q734: Answer = B
Processing row 735/1303...
✅ Processed Merge Q735: Answer = D
Processing row 736/1303...
✅ Processed Merge Q736: Answer = D
Processing row 737/1303...
✅ Processed Merge Q737: Answer = D
Processing row 738/1303...
✅ Processed Merge Q738: Answer =

✅ Processed Merge Q848: Answer = C
Processing row 849/1303...
✅ Processed Merge Q849: Answer = C
Processing row 850/1303...
✅ Processed Merge Q850: Answer = B
Saved progress to CSV after 850 items
Processing row 851/1303...
✅ Processed Merge Q851: Answer = G
Processing row 852/1303...
✅ Processed Merge Q852: Answer = A
Processing row 853/1303...
✅ Processed Merge Q853: Answer = D
Processing row 854/1303...
✅ Processed Merge Q854: Answer = B
Processing row 855/1303...
✅ Processed Merge Q855: Answer = B
Processing row 856/1303...
✅ Processed Merge Q856: Answer = B
Processing row 857/1303...
✅ Processed Merge Q857: Answer = B
Processing row 858/1303...
✅ Processed Merge Q858: Answer = C
Processing row 859/1303...
✅ Processed Merge Q859: Answer = D
Processing row 860/1303...
✅ Processed Merge Q860: Answer = C
Saved progress to CSV after 860 items
Processing row 861/1303...
✅ Processed Merge Q861: Answer = C
Processing row 862/1303...
✅ Processed Merge Q862: Answer = D
Processing row 863/13

✅ Processed Merge Q971: Answer = D
Processing row 972/1303...
✅ Processed Merge Q972: Answer = C
Processing row 973/1303...
✅ Processed Merge Q973: Answer = B
Processing row 974/1303...
✅ Processed Merge Q974: Answer = C
Processing row 975/1303...
✅ Processed Merge Q975: Answer = D
Processing row 976/1303...
✅ Processed Merge Q976: Answer = A
Processing row 977/1303...
✅ Processed Merge Q977: Answer = D
Processing row 978/1303...
✅ Processed Merge Q978: Answer = A
Processing row 979/1303...
✅ Processed Merge Q979: Answer = A
Processing row 980/1303...
✅ Processed Merge Q980: Answer = B
Saved progress to CSV after 980 items
Processing row 981/1303...
✅ Processed Merge Q981: Answer = D
Processing row 982/1303...
✅ Processed Merge Q982: Answer = A
Processing row 983/1303...
✅ Processed Merge Q983: Answer = C
Processing row 984/1303...
✅ Processed Merge Q984: Answer = B
Processing row 985/1303...
✅ Processed Merge Q985: Answer = B
Processing row 986/1303...
✅ Processed Merge Q986: Answer =

✅ Processed Merge Q1091: Answer = J
Processing row 1092/1303...
✅ Processed Merge Q1092: Answer = A
Processing row 1093/1303...
✅ Processed Merge Q1093: Answer = I
Processing row 1094/1303...
✅ Processed Merge Q1094: Answer = A
Processing row 1095/1303...
✅ Processed Merge Q1095: Answer = A
Processing row 1096/1303...
✅ Processed Merge Q1096: Answer = D
Processing row 1097/1303...
✅ Processed Merge Q1097: Answer = C
Processing row 1098/1303...
✅ Processed Merge Q1098: Answer = B
Processing row 1099/1303...
✅ Processed Merge Q1099: Answer = C
Processing row 1100/1303...
✅ Processed Merge Q1100: Answer = D
Saved progress to CSV after 1100 items
Processing row 1101/1303...
✅ Processed Merge Q1101: Answer = C
Processing row 1102/1303...
✅ Processed Merge Q1102: Answer = B
Processing row 1103/1303...
✅ Processed Merge Q1103: Answer = B
Processing row 1104/1303...
✅ Processed Merge Q1104: Answer = B
Processing row 1105/1303...
✅ Processed Merge Q1105: Answer = D
Processing row 1106/1303...
✅

✅ Processed Merge Q1212: Answer = A
Processing row 1213/1303...
✅ Processed Merge Q1213: Answer = D
Processing row 1214/1303...
✅ Processed Merge Q1214: Answer = A
Processing row 1215/1303...
✅ Processed Merge Q1215: Answer = D
Processing row 1216/1303...
✅ Processed Merge Q1216: Answer = C
Processing row 1217/1303...
✅ Processed Merge Q1217: Answer = A
Processing row 1218/1303...
✅ Processed Merge Q1218: Answer = A
Processing row 1219/1303...
✅ Processed Merge Q1219: Answer = E
Processing row 1220/1303...
✅ Processed Merge Q1220: Answer = C
Saved progress to CSV after 1220 items
Processing row 1221/1303...
✅ Processed Merge Q1221: Answer = I
Processing row 1222/1303...
✅ Processed Merge Q1222: Answer = C
Processing row 1223/1303...
✅ Processed Merge Q1223: Answer = E
Processing row 1224/1303...
✅ Processed Merge Q1224: Answer = G
Processing row 1225/1303...
✅ Processed Merge Q1225: Answer = C
Processing row 1226/1303...
✅ Processed Merge Q1226: Answer = B
Processing row 1227/1303...
✅

In [3]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv(paths.PREDICTIONS / "[SR]_Qwen_14B_predictions_Qwen14B.csv")
# df = pd.read_csv(paths.DATA / "After_Removal_High_qwen_72B_predictions.csv")

# # Ensure both dataframes have the same length
# assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == output_df['answer_corr']).astype(int)

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Analysis by category (assuming data_source_corr is in one of the dataframes)
# Check which dataframe has data_source_corr
if 'data_source_corr' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_corr' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_corr'] = df['data_source_corr']
else:
    print("Warning: 'data_source_corr' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_corr' in analysis_df.columns:
    category_stats = analysis_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

OVERALL ACCURACY ANALYSIS
Accuracy: 0.5081 (50.81%)
Standard Deviation: 0.5001
95% Confidence Interval: [0.4809, 0.5352]
95% CI (percentage): [48.09%, 53.52%]
Sample Size: 1303

CATEGORY-WISE ACCURACY ANALYSIS
data_source_corr  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
            jama    582       0.549828 0.497939 0.020640     0.509290     0.590367        54.982818  49.793892      50.928962      59.036674
      medbullets    207       0.560386 0.497543 0.034582     0.492207     0.628566        56.038647  49.754333      49.220713      62.856581
        medxpert    318       0.226415 0.419170 0.023506     0.180168     0.272662        22.641509  41.917040      18.016779      27.266240
            mmlu    193       0.797927 0.402590 0.028979     0.740769     0.855086        79.792746  40.259038      74.076921      85.508572


SUMMARY TABLE
            Metric           Value
  Overall Accuracy 0.5081 (50.81%)